# Lightweight Fine-Tuning Project

TODO: In this cell, describe your choices for each of the following

* PEFT technique: LoRA
* Model: GPT-2
* Evaluation approach: accuracy
* Fine-tuning dataset: imbd

In [57]:
!pip install transformers datasets evaluate accelerate peft
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import load_dataset
import numpy as np
import evaluate

In [58]:
pip install bitsandbytes

## Loading and Evaluating a Foundation Model

TODO: In the cells below, load your chosen pre-trained Hugging Face model and evaluate its performance prior to fine-tuning. This step includes loading an appropriate tokenizer and dataset.

In [ ]:
try:
    import evaluate
except ImportError:
    !pip install -q transformers datasets evaluate accelerate

from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import load_dataset
import numpy as np
import evaluate

#define model and dataset
MODEL_NAME = "gpt2"
DATASET_NAME = "imdb"

In [ ]:
#load Dataset and Tokenizer
dataset = load_dataset(DATASET_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

#create evaluation subset
test_subset = dataset["test"].shuffle(seed=42).select(range(500))
tokenized_test = test_subset.map(preprocess_function, batched=True)

In [ ]:
#load Foundation Model for Sequence Classification
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "NEGATIVE", 1: "POSITIVE"},
    label2id={"NEGATIVE": 0, "POSITIVE": 1}
)

#align model padding with tokenizer
model.config.pad_token_id = model.config.eos_token_id

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2ForSequenceClassification LOAD REPORT from: gpt2
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
#set up Evaluation Metric
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

In [ ]:
import torch
from torch.utils.data import DataLoader

#prepare the DataLoader
tokenized_test.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
baseline_dataloader = DataLoader(tokenized_test, batch_size=8)

#set model to evaluation mode and move to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

all_preds = []
all_labels = []

for batch in baseline_dataloader:
    batch = {k: v.to(device) for k, v in batch.items()}
    
    with torch.no_grad():
        outputs = model(**batch)
    
    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)
    
    all_preds.extend(predictions.cpu().numpy())
    all_labels.extend(batch["label"].cpu().numpy())

#calculate Accuracy and Store in baseline_results
baseline_accuracy = accuracy_metric.compute(predictions=all_preds, references=all_labels)['accuracy']
baseline_results = {'eval_accuracy': baseline_accuracy}

print(f"Foundation Model Accuracy: {baseline_results['eval_accuracy']:.2%}")

Evaluating baseline foundation model (Manual Loop)...
Foundation Model Accuracy: 51.00%


## Performing Parameter-Efficient Fine-Tuning

TODO: In the cells below, create a PEFT model from your loaded model, run a training loop, and save the PEFT model weights

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

#define LoRA Config
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8, 
    lora_alpha=32, 
    lora_dropout=0.1, 
    target_modules=["c_attn"],
    fan_in_fan_out=True
)

#wrap the model
peft_model = get_peft_model(model, peft_config)
peft_model.print_trainable_parameters()

trainable params: 296,448 || all params: 124,737,792 || trainable%: 0.2377


In [ ]:
#create a small training subset
train_subset = dataset["train"].shuffle(seed=42).select(range(1000))

def preprocess_for_peft(examples):
    result = tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)
    result["labels"] = examples["label"]
    return result

tokenized_train = train_subset.map(preprocess_for_peft, batched=True)

In [66]:
training_args = TrainingArguments(
    output_dir="./gpt2-lora-results",
    learning_rate=2e-4, 
    per_device_train_batch_size=4,
    num_train_epochs=1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test, 
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

c:\Users\shiva\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.538617,0.750000


TrainOutput(global_step=250, training_loss=0.760995361328125, metrics={'train_runtime': 4060.2614, 'train_samples_per_second': 0.246, 'train_steps_per_second': 0.062, 'total_flos': 262207438848000.0, 'train_loss': 0.760995361328125, 'epoch': 1.0})

In [ ]:
#save fine-tuned adapter weights
peft_model.save_pretrained("gpt2-lora-adapter")

PEFT Model saved successfully!


###  ⚠️ IMPORTANT ⚠️

Due to workspace storage constraints, you should not store the model weights in the same directory but rather use `/tmp` to avoid workspace crashes which are irrecoverable.
Ensure you save it in /tmp always.

In [ ]:
#save model directory
peft_model.save_pretrained("/tmp/peft-model")

## Performing Inference with a PEFT Model

TODO: In the cells below, load the saved PEFT model weights and evaluate the performance of the trained PEFT model. Be sure to compare the results to the results from prior to fine-tuning.

In [ ]:
from peft import PeftModel, PeftConfig

#define path for weights
peft_model_path = "/tmp/peft-model"

In [ ]:
#load base model
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "NEGATIVE", 1: "POSITIVE"},
    label2id={"NEGATIVE": 0, "POSITIVE": 1}
)

#PT-2 padding alignment
base_model.config.pad_token_id = base_model.config.eos_token_id

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2ForSequenceClassification LOAD REPORT from: gpt2
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
#load the PEFT model by combining the base model with the saved adapters
loaded_peft_model = PeftModel.from_pretrained(base_model, peft_model_path)

PEFT weights loaded successfully from /tmp/peft-model!


In [ ]:
import torch
from torch.utils.data import DataLoader

#evaluate dataloader
tokenized_test.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
eval_dataloader = DataLoader(tokenized_test, batch_size=8)

#set model to eval mode
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loaded_peft_model.to(device)
loaded_peft_model.eval()

all_preds = []
all_labels = []

print("Running inference with PEFT model...")
for batch in eval_dataloader:
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        outputs = loaded_peft_model(**batch)
    
    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)
    
    all_preds.extend(predictions.cpu().numpy())
    all_labels.extend(batch["label"].cpu().numpy())

Running inference with PEFT model...


In [ ]:
#calcualte Final Accuracy using the same metric as the baseline
peft_accuracy = accuracy_metric.compute(predictions=all_preds, references=all_labels)['accuracy']

#Print results
print("Results")
# baseline_results from first evalueation
print(f"Foundation Model Accuracy: {baseline_results['eval_accuracy']:.2%}")
print(f"Fine-tuned PEFT Model Accuracy: {peft_accuracy:.2%}")

improvement = peft_accuracy - baseline_results['eval_accuracy']
print(f"Total Improvement: {improvement:.2%}")

if improvement > 0:
    print("Success! The PEFT fine-tuning improved the model performance.")
else:
    print("Check your training hyperparameters; the model did not show improvement.")

--- Final Project Results ---
Foundation Model Accuracy: 51.00%
Fine-tuned PEFT Model Accuracy: 75.00%
Total Improvement: 24.00%
Success! The PEFT fine-tuning improved the model performance.
